# 스캔 PDF 페이지 분류 PoC — Colab 실행

위에서부터 차례로 실행하세요 (**런타임 → 모두 실행**). GPU(T4)가 있으면 빠르지만, 없어도 돌아갑니다.

샘플은 전부 가짜 서류입니다. 실제 고객 서류는 이 노트북에 올리지 마세요.

In [ ]:
!git clone -q https://github.com/weriousdf/scan-pdf-sorter-poc.git
%cd scan-pdf-sorter-poc
!pip install -q pymupdf

In [ ]:
import torch, transformers, pymupdf
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 (CPU)')
print('torch', torch.__version__, '| transformers', transformers.__version__, '| pymupdf', pymupdf.__version__)

## 1. 기준선 — 규칙 (잉크 영역 크기)

In [ ]:
!python poc/split_pdf.py --pdfs data/samples --backend rule --run-id colab
!python poc/score.py --truth data/ground_truth.csv --pred outputs/colab/rule/predictions.csv --sweep

## 2. AI — CLIP 제로샷 분류
처음 실행할 때 모델(약 600MB)을 내려받습니다.

In [ ]:
!python poc/split_pdf.py --pdfs data/samples --backend clip --run-id colab
!python poc/score.py --truth data/ground_truth.csv --pred outputs/colab/clip/predictions.csv --sweep

## 3. 결과 폴더 확인

In [ ]:
!find outputs/colab/clip -name '*.jpg' | sort

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt, glob
files = sorted(glob.glob('outputs/colab/clip/*/*.jpg'))
fig, axes = plt.subplots(3, 7, figsize=(21, 10))
for ax in axes.flat: ax.axis('off')
for ax, f in zip(axes.flat, files):
    ax.imshow(Image.open(f)); ax.set_title(f.split('/')[-2] + '
' + f.split('/')[-1], fontsize=8)
plt.tight_layout(); plt.show()

## 4. 결과 내려받기
`outputs_colab.zip` 을 내려받아 저장소의 `outputs/` 에 풀면 됩니다.

In [ ]:
!cd outputs && zip -qr ../outputs_colab.zip colab
from google.colab import files
files.download('outputs_colab.zip')